In [ ]:
from flask import Flask, request, jsonify, session
from flask_cors import CORS
import pandas as pd
import os
import requests
import json
import traceback
import hashlib
import random
import time
from datetime import datetime
from collections import defaultdict
import re

# ----------------------------
# Flask Setup
# ----------------------------

app = Flask(__name__)
app.secret_key = 'your-secret-key-here-change-in-production'
CORS(app, origins=["http://localhost:3000", "http://127.0.0.1:3000"])

# ----------------------------
# Configuration
# ----------------------------

FILE_PATH = r"C:\Users\hp\Desktop\imsi_20260227_101529.csv"
OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "tinyllama"

# ----------------------------
# Load Data Function (ADD THIS AT THE TOP)
# ----------------------------

def load_data():
    """Load CSV data"""
    print(f"📁 Checking file: {FILE_PATH}")
    
    if not os.path.exists(FILE_PATH):
        print(f"❌ File not found: {FILE_PATH}")
        return pd.DataFrame()
    
    try:
        data = pd.read_csv(FILE_PATH, low_memory=False)
        print(f"✅ Loaded {len(data)} rows")
        print(f"📊 Columns: {data.columns.tolist()}")
        return data
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        traceback.print_exc()
        return pd.DataFrame()

# ----------------------------
# Helper Functions (ADD THESE BEFORE THEY'RE CALLED)
# ----------------------------

def prepare_data_context(data, question):
    """Prepare data context for AI"""
    context_parts = []
    context_parts.append(f"Dataset has {len(data)} records")
    
    if 'operator' in data.columns:
        operators = data['operator'].value_counts().head(3).index.tolist()
        context_parts.append(f"Operators: {', '.join(operators)}")
    
    if 'imsi' in data.columns:
        context_parts.append(f"Unique users: {data['imsi'].nunique()}")
    
    return "\n".join(context_parts)

def clean_ai_response(response, data, question):
    """Clean AI response"""
    if response and len(response) > 0 and response[0] in ['📊', '📱', '👥', '🌍', '⏰', '🏷️', '💡']:
        return response
    
    # Remove generic phrases
    generic_phrases = ["Sure!", "Based on", "Here's", "I understand", "Certainly!"]
    for phrase in generic_phrases:
        response = response.replace(phrase, "")
    
    if len(response.strip()) < 30:
        response = extract_direct_data(data, question)
    
    return response.strip()

# ----------------------------
# Intelligent Learning System
# ----------------------------

class IntelligentLearningSystem:
    def __init__(self, user_id="default"):
        self.user_id = user_id
        self.memory_file = f"user_memory_{user_id}.json"
        
        # Load or create user profile
        self.user_profile = self.load_memory()
        
        # Conversation context
        self.conversation_history = []
        self.last_response = None
        
    def load_memory(self):
        """Load user-specific learned data"""
        if os.path.exists(self.memory_file):
            try:
                with open(self.memory_file, 'r') as f:
                    data = json.load(f)
                    # Convert lists back to sets if needed
                    if "vocabulary" in data and isinstance(data["vocabulary"], list):
                        data["vocabulary"] = set(data["vocabulary"])
                    # Convert dicts back to defaultdicts
                    if "preferences" in data and "topics" in data["preferences"]:
                        data["preferences"]["topics"] = defaultdict(int, data["preferences"]["topics"])
                    if "patterns" in data:
                        if "question_types" in data["patterns"]:
                            data["patterns"]["question_types"] = defaultdict(int, data["patterns"]["question_types"])
                        if "time_patterns" in data["patterns"]:
                            data["patterns"]["time_patterns"] = defaultdict(int, data["patterns"]["time_patterns"])
                    return data
            except Exception as e:
                print(f"Error loading memory: {e}")
                return self.create_default_profile()
        else:
            return self.create_default_profile()
    
    def create_default_profile(self):
        """Create default user profile"""
        return {
            "user_info": {
                "name": None,
                "preferences": {},
                "first_seen": str(datetime.now())
            },
            "knowledge": {
                "facts": {},
                "corrections": [],
                "user_mistakes": [],
                "verified_info": {}
            },
            "preferences": {
                "response_style": "balanced",
                "preferred_length": "balanced",
                "emoji_preference": 0,
                "topics": defaultdict(int),
                "feedback_history": []
            },
            "learning": {
                "conversation_count": 0,
                "last_learning": str(datetime.now()),
                "adaptation_level": 0.0
            },
            "vocabulary": set(),
            "patterns": {
                "question_types": defaultdict(int),
                "time_patterns": defaultdict(int),
                "common_phrases": defaultdict(int)
            }
        }
    
    def save_memory(self):
        """Save learned data"""
        try:
            # Convert defaultdicts and sets to regular types for JSON
            profile_copy = {
                "user_info": self.user_profile["user_info"],
                "knowledge": self.user_profile["knowledge"],
                "preferences": {
                    **self.user_profile["preferences"],
                    "topics": dict(self.user_profile["preferences"]["topics"])
                },
                "learning": self.user_profile["learning"],
                "vocabulary": list(self.user_profile["vocabulary"]),
                "patterns": {
                    "question_types": dict(self.user_profile["patterns"]["question_types"]),
                    "time_patterns": dict(self.user_profile["patterns"]["time_patterns"]),
                    "common_phrases": dict(self.user_profile["patterns"]["common_phrases"])
                }
            }
            
            with open(self.memory_file, 'w') as f:
                json.dump(profile_copy, f, default=str, indent=2)
        except Exception as e:
            print(f"Error saving memory: {e}")
    
    def learn_from_interaction(self, question, response, user_feedback=None):
        """Main learning function"""
        try:
            self.user_profile["learning"]["conversation_count"] += 1
            self.learn_question_pattern(question)
            self.learn_user_preferences(question)
            self.extract_and_learn_facts(question, response)
            self.learn_vocabulary(question)
            
            self.conversation_history.append({
                "question": question,
                "response": response,
                "timestamp": str(datetime.now()),
                "feedback": user_feedback
            })
            
            self.conversation_history = self.conversation_history[-50:]
            self.update_adaptation_level()
            self.save_memory()
        except Exception as e:
            print(f"Error in learn_from_interaction: {e}")
    
    def learn_question_pattern(self, question):
        """Learn what types of questions user asks"""
        try:
            hour = datetime.now().hour
            if hour < 12:
                time_period = "morning"
            elif hour < 17:
                time_period = "afternoon"
            else:
                time_period = "evening"
            
            self.user_profile["patterns"]["time_patterns"][time_period] += 1
            
            if "?" in question:
                if any(word in question.lower() for word in ["what", "tell", "show"]):
                    q_type = "information"
                elif any(word in question.lower() for word in ["how", "why", "when"]):
                    q_type = "explanation"
                elif any(word in question.lower() for word in ["help", "what can"]):
                    q_type = "help"
                else:
                    q_type = "general"
                
                self.user_profile["patterns"]["question_types"][q_type] += 1
        except Exception as e:
            print(f"Error in learn_question_pattern: {e}")
    
    def learn_user_preferences(self, question):
        """Learn user's communication style"""
        try:
            question_lower = question.lower()
            
            formal_words = ["please", "thank you", "appreciate", "would you", "could you"]
            casual_words = ["bro", "dude", "lol", "haha", "yeah", "hey"]
            
            if any(word in question_lower for word in formal_words):
                self.user_profile["preferences"]["formality_score"] = self.user_profile["preferences"].get("formality_score", 0) + 1
            if any(word in question_lower for word in casual_words):
                self.user_profile["preferences"]["casual_score"] = self.user_profile["preferences"].get("casual_score", 0) + 1
            
            if any(emoji in question for emoji in ["😊", "😂", "👍", "❤️", "🎉"]):
                self.user_profile["preferences"]["emoji_preference"] = min(10, self.user_profile["preferences"]["emoji_preference"] + 1)
            
            topics = self.extract_topics(question)
            for topic in topics:
                self.user_profile["preferences"]["topics"][topic] += 1
        except Exception as e:
            print(f"Error in learn_user_preferences: {e}")
    
    def extract_topics(self, text):
        """Extract topics from text"""
        topics = []
        topic_keywords = {
            "operator": ["operator", "network", "carrier", "provider", "telco"],
            "device": ["device", "imsi", "imei", "phone", "mobile", "user"],
            "country": ["country", "location", "region", "area"],
            "time": ["time", "date", "when", "period", "timestamp"],
            "brand": ["brand", "make", "manufacturer", "model"],
            "signal": ["signal", "strength", "rssi", "quality", "coverage"]
        }
        
        for topic, keywords in topic_keywords.items():
            if any(keyword in text.lower() for keyword in keywords):
                topics.append(topic)
        
        return topics if topics else ["general"]
    
    def extract_and_learn_facts(self, question, response):
        """Extract and learn facts from conversation"""
        try:
            if "top operator" in response.lower() or "main operator" in response.lower():
                match = re.search(r'[•*]\s*\*\*([^*]+)\*\*', response)
                if match:
                    operator = match.group(1).strip()
                    fact_hash = hashlib.md5(f"operator:{operator}".encode()).hexdigest()
                    self.user_profile["knowledge"]["facts"][fact_hash] = {
                        "fact": f"Main operator is {operator}",
                        "confidence": 0.8,
                        "source": "data_analysis",
                        "timestamp": str(datetime.now()),
                        "verified": True
                    }
        except Exception as e:
            print(f"Error in extract_and_learn_facts: {e}")
    
    def learn_vocabulary(self, question):
        """Build vocabulary from user"""
        try:
            words = re.findall(r'\b[a-zA-Z]{3,}\b', question.lower())
            for word in words:
                self.user_profile["vocabulary"].add(word)
        except Exception as e:
            print(f"Error in learn_vocabulary: {e}")
    
    def update_adaptation_level(self):
        """Calculate how well adapted the bot is to user"""
        try:
            conv_count = self.user_profile["learning"]["conversation_count"]
            
            if conv_count < 10:
                adaptation = conv_count / 10
            elif conv_count < 50:
                adaptation = 0.8
            else:
                adaptation = 0.95
            
            self.user_profile["learning"]["adaptation_level"] = adaptation
        except Exception as e:
            print(f"Error in update_adaptation_level: {e}")
    
    def detect_correction(self, message, previous_message=None):
        """Detect if user is correcting the bot"""
        correction_patterns = [
            r"actually[,]?\s+(.+)",
            r"no[,]?\s+it[']?s\s+(.+)",
            r"correction[,]?\s+(.+)",
            r"wait[,]?\s+it[']?s\s+(.+)",
        ]
        
        for pattern in correction_patterns:
            match = re.search(pattern, message.lower())
            if match:
                correction_text = match.group(1) if match.groups() else message
                return {
                    "is_correction": True,
                    "correction": correction_text,
                    "original_context": previous_message
                }
        
        return {"is_correction": False}
    
    def learn_from_correction(self, correction_data):
        """Update knowledge based on user correction"""
        if correction_data["is_correction"]:
            self.user_profile["knowledge"]["corrections"].append({
                "correction": correction_data["correction"],
                "original": correction_data["original_context"],
                "timestamp": str(datetime.now())
            })
            return True
        return False
    
    def adapt_response(self, response, question):
        """Adapt response based on learned user preferences"""
        try:
            adaptation_level = self.user_profile["learning"]["adaptation_level"]
            conv_count = self.user_profile["learning"]["conversation_count"]
            
            # Add user's name if known
            if self.user_profile["user_info"].get("name") and random.random() < 0.3:
                response = f"Hey {self.user_profile['user_info']['name']}! {response}"
            
            # Add emojis if user likes them
            emoji_pref = self.user_profile["preferences"]["emoji_preference"]
            if emoji_pref > 5 and random.random() < 0.4:
                response += random.choice([" 😊", " 👍", " ✨", " 📊", " 🎯"])
            
            return response
        except Exception as e:
            print(f"Error in adapt_response: {e}")
            return response
    
    def make_casual(self, text):
        """Make text more casual"""
        casual_replacements = {
            "hello": "hey",
            "yes": "yeah",
            "I am": "I'm",
            "you are": "you're",
        }
        
        for formal, casual in casual_replacements.items():
            text = text.replace(formal, casual)
        
        return text
    
    def personalize_with_memory(self, question):
        """Add personal context based on memory"""
        context = ""
        conv_count = self.user_profile["learning"]["conversation_count"]
        
        if conv_count > 20:
            topics = self.extract_topics(question)
            for topic in topics:
                if self.user_profile["preferences"]["topics"].get(topic, 0) > 3:
                    context = f"You've asked about {topic}s before, so here's what I found: "
                    break
        
        return context
    
    def get_learning_status(self):
        """Get current learning status"""
        return {
            "conversations": self.user_profile["learning"]["conversation_count"],
            "adaptation_level": self.user_profile["learning"]["adaptation_level"],
            "known_topics": list(self.user_profile["preferences"]["topics"].keys())[:5],
            "vocabulary_size": len(self.user_profile["vocabulary"])
        }

# ----------------------------
# Enhanced Ollama Client
# ----------------------------

class EnhancedOllamaClient:
    def __init__(self, base_url=OLLAMA_URL, model=MODEL_NAME, learning_system=None):
        self.base_url = base_url.rstrip('/')
        self.model = model
        self.headers = {"Content-Type": "application/json"}
        self.learning_system = learning_system
        
    def check_connection(self):
        """Check if Ollama server is running"""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            return response.status_code == 200
        except:
            return False
    
    def generate_with_personality(self, prompt, user_question, data_context):
        """Generate response with learned personality"""
        try:
            personal_context = ""
            if self.learning_system:
                personal_context = self.learning_system.personalize_with_memory(user_question)
            
            full_prompt = f"""{personal_context}
DATASET CONTEXT:
{data_context}

USER: {user_question}

Provide a helpful, friendly answer with specific numbers from the data:"""
            
            payload = {
                "model": self.model,
                "prompt": full_prompt,
                "stream": False,
                "options": {
                    "temperature": 0.7,
                    "num_predict": 400,
                    "num_ctx": 1024,
                }
            }
            
            response = requests.post(
                f"{self.base_url}/api/generate",
                headers=self.headers,
                data=json.dumps(payload),
                timeout=60
            )
            
            if response.status_code == 200:
                result = response.json()
                return result.get("response", "No response generated")
            else:
                return None
                
        except Exception as e:
            print(f"Ollama error: {e}")
            return None

# ----------------------------
# Data Extraction Function
# ----------------------------

def extract_direct_data(data, question, learning_system=None):
    """Extract data with personalized formatting"""
    
    question_lower = question.lower()
    
    # Check for name learning
    if "my name is" in question_lower:
        name = question_lower.split("my name is")[-1].strip()
        if learning_system:
            learning_system.user_profile["user_info"]["name"] = name.title()
            learning_system.save_memory()
        return f"🎉 Nice to meet you, {name.title()}! I'll remember that. How can I help you today?"
    
    # Operator questions
    if any(word in question_lower for word in ['operator', 'operators', 'network', 'carrier']):
        if 'operator' in data.columns:
            operator_counts = data['operator'].value_counts()
            
            personal_greeting = ""
            if learning_system and learning_system.user_profile["user_info"].get("name"):
                personal_greeting = f"Hey {learning_system.user_profile['user_info']['name']}, "
            
            response = [f"{personal_greeting}📱 **Network Operators Found:**"]
            for operator, count in operator_counts.head(10).items():
                percentage = (count / len(data)) * 100
                response.append(f"• **{operator}**: {count} records ({percentage:.1f}%)")
            return "\n".join(response)
    
    # User/Device questions
    elif any(word in question_lower for word in ['user', 'users', 'device', 'devices', 'imsi']):
        if 'imsi' in data.columns:
            total_users = data['imsi'].nunique()
            response = ["👥 **User Information:**"]
            response.append(f"• **Total unique users**: {total_users}")
            return "\n".join(response)
    
    # Help command
    elif any(word in question_lower for word in ['help', 'what can', 'how to', 'commands']):
        response = ["💡 **Here's what I can help with:**"]
        response.append("")
        response.append("📱 **Operators**: 'Show me operators', 'What networks?'")
        response.append("👥 **Users**: 'Show me users', 'How many devices?'")
        response.append("🌍 **Countries**: 'What countries?', 'Location data'")
        response.append("📊 **Summary**: 'Give me overview', 'Dashboard'")
        response.append("")
        response.append("🎯 **Special Features:**")
        response.append("• Tell me your name: 'My name is [name]'")
        response.append("• I learn your preferences over time!")
        
        if learning_system and learning_system.user_profile["learning"]["conversation_count"] > 5:
            response.append("")
            response.append(f"📚 **Learning status**: I'm getting to know you! ({learning_system.user_profile['learning']['conversation_count']} conversations)")
        
        return "\n".join(response)
    
    # Summary
    elif any(word in question_lower for word in ['summary', 'overview', 'dashboard', 'stats']):
        response = ["📊 **Your Data Dashboard**"]
        response.append(f"• **Total Records**: {len(data)}")
        
        if 'imsi' in data.columns:
            response.append(f"• **Unique Users**: {data['imsi'].nunique()}")
        
        if 'operator' in data.columns:
            top_op = data['operator'].value_counts().index[0] if not data['operator'].empty else "Unknown"
            response.append(f"• **Main Operator**: {top_op}")
        
        if learning_system and learning_system.user_profile["user_info"].get("name"):
            response[0] = f"📊 **Hey {learning_system.user_profile['user_info']['name']}! Here's your dashboard**"
        
        return "\n".join(response)
    
    # Default response
    else:
        response = ["📊 **Quick Overview:**"]
        response.append(f"• **{len(data)} total records**")
        
        if 'imsi' in data.columns:
            response.append(f"• **{data['imsi'].nunique()} unique users**")
        
        if 'operator' in data.columns:
            top_op = data['operator'].value_counts().index[0] if not data['operator'].empty else "Unknown"
            response.append(f"• **Main operator**: {top_op}")
        
        response.append("")
        response.append("💡 **Type 'help' to see what I can do!**")
        response.append("💡 **Tell me your name for personalized responses!**")
        
        return "\n".join(response)

# ----------------------------
# Chat Route
# ----------------------------

@app.route("/chat", methods=["POST", "OPTIONS"])
def chat():
    if request.method == "OPTIONS":
        response = jsonify({"status": "ok"})
        response.headers.add("Access-Control-Allow-Origin", "*")
        response.headers.add("Access-Control-Allow-Headers", "Content-Type")
        response.headers.add("Access-Control-Allow-Methods", "POST")
        return response
    
    try:
        req = request.get_json()
        print("="*50)
        print(f"📨 Request: {req}")
        
        if not req or "message" not in req:
            return jsonify({"reply": "Please type a question."})
        
        question = req["message"]
        user_id = req.get("user_id", "default")
        
        print(f"❓ Question: {question}")
        
        # Create learning system
        learning_system = IntelligentLearningSystem(user_id)
        
        # Load data
        data = load_data()
        
        if data.empty:
            return jsonify({
                "reply": f"⚠️ No data found at: {FILE_PATH}\n\nPlease check the file path."
            })
        
        # Simulate thinking
        time.sleep(random.uniform(0.3, 0.8))
        
        # Get direct response
        direct_response = extract_direct_data(data, question, learning_system)
        
        # Learn from interaction
        learning_system.learn_from_interaction(question, direct_response)
        
        # Adapt response
        adapted_response = learning_system.adapt_response(direct_response, question)
        learning_system.last_response = adapted_response
        
        print(f"✅ Response sent")
        print("="*50)
        
        return jsonify({
            "reply": adapted_response,
            "status": "success",
            "source": "direct",
            "learning_status": learning_system.get_learning_status()
        })
        
    except Exception as e:
        print(f"❌ ERROR: {e}")
        traceback.print_exc()
        return jsonify({
            "reply": f"Error: {str(e)}",
            "status": "error"
        })

# ----------------------------
# Test Route
# ----------------------------

@app.route("/test", methods=["GET"])
def test():
    """Test endpoint"""
    data_exists = os.path.exists(FILE_PATH)
    data_info = {}
    
    if data_exists:
        try:
            data = pd.read_csv(FILE_PATH, nrows=3)
            data_info = {
                "rows_preview": len(data),
                "columns": data.columns.tolist()[:10]
            }
        except Exception as e:
            data_info = {"error": str(e)}
    
    return jsonify({
        "status": "running",
        "data_file_exists": data_exists,
        "file_path": FILE_PATH,
        "data_info": data_info,
        "configured_model": MODEL_NAME
    })

@app.route("/learning-status", methods=["GET"])
def learning_status():
    """Get learning status"""
    user_id = request.args.get("user_id", "default")
    learning_system = IntelligentLearningSystem(user_id)
    return jsonify(learning_system.get_learning_status())

# ----------------------------
# Run Server
# ----------------------------

if __name__ == "__main__":
    print("="*60)
    print("🚀 CrowdSense AI Chatbot - WITH LEARNING!")
    print("="*60)
    print(f"📡 Server: http://127.0.0.1:5000")
    print(f"📁 Data file: {FILE_PATH}")
    print(f"🧠 Model: {MODEL_NAME}")
    print("="*60)
    
    # Test file loading on startup
    test_data = load_data()
    if not test_data.empty:
        print(f"✅ Data loaded: {len(test_data)} rows")
        print(f"📊 Columns: {', '.join(test_data.columns.tolist()[:5])}")
    else:
        print(f"❌ Could not load data from: {FILE_PATH}")
    
    print("="*60)
    print("📋 Test endpoint: GET /test")
    print("💬 Chat endpoint: POST /chat")
    print("="*60)
    
    app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False)

🚀 CrowdSense AI Chatbot - WITH LEARNING!
📡 Server: http://127.0.0.1:5000
📁 Data file: C:\Users\hp\Desktop\imsi_20260227_101529.csv
🧠 Model: tinyllama
📁 Checking file: C:\Users\hp\Desktop\imsi_20260227_101529.csv
✅ Loaded 92 rows
📊 Columns: ['index', 'tmsi1', 'tmsi2', 'imsi', 'country', 'brand', 'operator', 'mcc', 'mnc', 'lac', 'cellid', 'timestamp', 'capture_time']
✅ Data loaded: 92 rows
📊 Columns: index, tmsi1, tmsi2, imsi, country
📋 Test endpoint: GET /test
💬 Chat endpoint: POST /chat
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.100.228:5000
Press CTRL+C to quit
127.0.0.1 - - [24/Mar/2026 17:28:25] "OPTIONS /chat HTTP/1.1" 200 -


📨 Request: {'message': 'hi'}
❓ Question: hi
📁 Checking file: C:\Users\hp\Desktop\imsi_20260227_101529.csv
✅ Loaded 92 rows
📊 Columns: ['index', 'tmsi1', 'tmsi2', 'imsi', 'country', 'brand', 'operator', 'mcc', 'mnc', 'lac', 'cellid', 'timestamp', 'capture_time']


127.0.0.1 - - [24/Mar/2026 17:28:25] "POST /chat HTTP/1.1" 200 -


✅ Response sent


...

✅ Response sent


127.0.0.1 - - [24/Mar/2026 17:29:41] "OPTIONS /chat HTTP/1.1" 200 -


📨 Request: {'message': 'what my name'}
❓ Question: what my name
📁 Checking file: C:\Users\hp\Desktop\imsi_20260227_101529.csv
✅ Loaded 92 rows
📊 Columns: ['index', 'tmsi1', 'tmsi2', 'imsi', 'country', 'brand', 'operator', 'mcc', 'mnc', 'lac', 'cellid', 'timestamp', 'capture_time']


127.0.0.1 - - [24/Mar/2026 17:29:41] "POST /chat HTTP/1.1" 200 -


✅ Response sent
